# IFJC4002 | Practical Artificial Intelligence with Python
## Case 2: Used Car Resale Price Prediction (Regression)
**Faculty of Intelligent Computing, Universiti Malaysia Perlis (UniMAP)**  
**Course Coordinator:** Ir. Dr. Marni Azira Markom  
**Group 1**  

---
### 🎯 Project Overview & Rubric Alignment:
1. **Task 1: Problem Definition & Data Analysis (CO1):** Identify user, prediction objective (RM), input features, missing value audit, and statistical distributions.
2. **Task 2: Design, Develop & Evaluate Models (CO2):** Pipeline preprocessing (ColumnTransformer), Baseline model, algorithm comparison (Linear Regression, Ridge, Random Forest, Gradient Boosting) using 5-Fold Cross-Validation, and reporting MAE, RMSE, and R².
3. **Task 3: Implementation, Deployment & Model Export (CO3):** Export pipeline to `.joblib` in `models/`, verify test inference, and prepare for Streamlit integration.

### 1. Mount Google Drive & Environment Setup
Run the cell below to connect your Google Colab session to Google Drive and navigate to the project directory.

In [ ]:
import os
import sys

# Check if running inside Google Colab
try:
    import google.colab
    IN_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Set working directory to project_1_unimap
    project_dir = '/content/drive/MyDrive/project_1_unimap'
    if os.path.exists(project_dir):
        os.chdir(project_dir)
        print(f"✅ Successfully changed directory to: {os.getcwd()}")
    else:
        print(f"⚠️ Project directory not found at {project_dir}. Working in current folder: {os.getcwd()}")
except ImportError:
    IN_COLAB = False
    print("💻 Running in local environment.")

### 2. Import Libraries
Standard data science and machine learning packages.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# Plotting theme for high-quality report figures
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams['figure.dpi'] = 150
print("✅ Libraries imported successfully!")

### 3. Load Dataset
Loads the approved dataset `02_Used_Car_Price.xlsx` from `data/raw/` with automatic path fallback.

In [ ]:
candidate_paths = [
    'data/raw/02_Used_Car_Price.xlsx',
    '/content/drive/MyDrive/project_1_unimap/data/raw/02_Used_Car_Price.xlsx',
    '../data/raw/02_Used_Car_Price.xlsx',
    '02_Used_Car_Price.xlsx'
]

data_path = None
for p in candidate_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError("❌ Could not find 02_Used_Car_Price.xlsx. Please ensure Drive is mounted or upload the file to Colab.")

df = pd.read_excel(data_path)
print(f"✅ Dataset loaded from: {data_path}")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
display(df.head())

## 📊 Task 1: Exploratory Data Analysis & Problem Definition (CO1)
- **Target:** `resale_price_rm` (continuous, Malaysian Ringgit)
- **Excluded feature:** `record_id` (identifier)
- **Auditing:** Missing values, numeric distributions, categorical splits, and correlations.

In [ ]:
# Missing Values Audit
missing_summary = df.isnull().sum()
print("=== Missing Values Audit ===")
print(missing_summary[missing_summary > 0])

# Numerical Features Summary
num_cols = ['vehicle_age_years', 'mileage_km', 'engine_size_l', 'condition_rating', 'previous_owners', 'resale_price_rm']
print("\n=== Descriptive Statistics (Numerical Features) ===")
display(df[num_cols].describe().round(2))

# Categorical Features Summary
cat_cols = ['fuel_type', 'transmission', 'market_segment']
print("\n=== Categorical Value Counts ===")
for c in cat_cols:
    print(f"\n-- {c} --")
    print(df[c].value_counts())

In [ ]:
# Figure 1: Target Distribution & Market Segment Comparison (Save for Report)
os.makedirs('outputs', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution plot
sns.histplot(df['resale_price_rm'], kde=True, color='royalblue', ax=axes[0])
axes[0].set_title('Distribution of Resale Price (RM)', fontweight='bold')
axes[0].set_xlabel('Resale Price (RM)')
axes[0].set_ylabel('Frequency')

# Boxplot by Market Segment
sns.boxplot(x='market_segment', y='resale_price_rm', data=df, palette='Set2', ax=axes[1])
axes[1].set_title('Resale Price by Market Segment', fontweight='bold')
axes[1].set_xlabel('Market Segment')
axes[1].set_ylabel('Resale Price (RM)')

plt.tight_layout()
plt.savefig('outputs/eda_price_distribution.png', bbox_inches='tight')
plt.show()
print("💾 Saved outputs/eda_price_distribution.png")

In [ ]:
# Figure 2: Correlation Heatmap
plt.figure(figsize=(8, 6))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation with Resale Price (RM)', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/eda_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print("💾 Saved outputs/eda_correlation_heatmap.png")

## ⚙️ Task 2: Design, Preprocessing & Model Evaluation (CO2)
- Separate features ($X$) and target ($y$)
- Train/test split (80% train, 20% test, `random_state=42`)
- Preprocessing Pipeline with `ColumnTransformer` (fit on train only to prevent leakage)
- Baseline: Training Mean Predictor
- Candidate Algorithms: Linear Regression, Ridge, Random Forest, Gradient Boosting
- Metrics: **MAE**, **RMSE**, and **R²**

In [ ]:
# Feature Selection & Data Splitting
feature_cols = [
    'vehicle_age_years', 'mileage_km', 'engine_size_l',
    'fuel_type', 'transmission', 'market_segment',
    'condition_rating', 'previous_owners'
]
X = df[feature_cols]
y = df['resale_price_rm']

# 80/20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set : {X_train.shape[0]} samples")
print(f"Testing set  : {X_test.shape[0]} samples")

# Construct Preprocessing Pipeline
numeric_features = ['vehicle_age_years', 'mileage_km', 'engine_size_l', 'condition_rating', 'previous_owners']
categorical_features = ['fuel_type', 'transmission', 'market_segment']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])
print("✅ Preprocessing pipeline configured successfully!")

In [ ]:
# 1. Baseline Model (Predicting Training Mean)
y_train_mean = y_train.mean()
y_base_pred = np.full_like(y_test, fill_value=y_train_mean)

mae_base = mean_absolute_error(y_test, y_base_pred)
rmse_base = root_mean_squared_error(y_test, y_base_pred)
r2_base = r2_score(y_test, y_base_pred)

print("=" * 50)
print("BASELINE MODEL EVALUATION (Training Mean)")
print(f"Training Mean : RM {y_train_mean:,.2f}")
print(f"Test MAE      : RM {mae_base:,.2f}")
print(f"Test RMSE     : RM {rmse_base:,.2f}")
print(f"Test R²       : {r2_base:.4f}")
print("=" * 50)

In [ ]:
# 2. Model Training, 5-Fold Cross Validation & Test Evaluation
candidate_models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting Regressor': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = [
    {
        'Model': 'Baseline (Mean)',
        '5-Fold CV R²': 0.0,
        'Test MAE (RM)': round(mae_base, 2),
        'Test RMSE (RM)': round(rmse_base, 2),
        'Test R²': round(r2_base, 4)
    }
]

trained_pipelines = {}
cv = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in candidate_models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # 5-Fold Cross Validation on Train Split
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='r2')
    cv_r2 = cv_scores.mean()
    
    # Fit on Train Split, Test on Reserved Test Set
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        '5-Fold CV R²': round(cv_r2, 4),
        'Test MAE (RM)': round(mae, 2),
        'Test RMSE (RM)': round(rmse, 2),
        'Test R²': round(r2, 4)
    })
    trained_pipelines[name] = pipe

# Display Model Comparison Table
comparison_df = pd.DataFrame(results)
print("\n" + "=" * 65)
print("MODEL COMPARISON TABLE (Mandated by Rubric)")
print("=" * 65)
display(comparison_df)
comparison_df.to_csv('outputs/model_comparison.csv', index=False)
print("💾 Saved outputs/model_comparison.csv")

In [ ]:
# Figure 3: Actual vs Predicted & Residuals Plot for Best Model
best_model_name = comparison_df.sort_values(by='Test R²', ascending=False).iloc[0]['Model']
best_pipe = trained_pipelines[best_model_name]
y_best_pred = best_pipe.predict(X_test)
residuals = y_test - y_best_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_best_pred, alpha=0.5, color='darkblue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_title(f'Actual vs. Predicted Resale Price ({best_model_name})', fontweight='bold')
axes[0].set_xlabel('Actual Price (RM)')
axes[0].set_ylabel('Predicted Price (RM)')

# Residuals distribution
sns.histplot(residuals, kde=True, color='crimson', ax=axes[1])
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_title('Residuals Distribution', fontweight='bold')
axes[1].set_xlabel('Residual (Actual - Predicted) [RM]')

plt.tight_layout()
plt.savefig('outputs/model_residuals_plot.png', bbox_inches='tight')
plt.show()
print("💾 Saved outputs/model_residuals_plot.png")

## 🚀 Task 3: Implementation, Deployment & Monitoring (CO3)
- Export complete pipeline to `models/car_price_pipeline.joblib`
- Verify reloading and predict on test sample
- Prepare Streamlit app demonstration

In [ ]:
# Save Selected Model Pipeline
os.makedirs('models', exist_ok=True)
model_export_path = 'models/car_price_pipeline.joblib'
joblib.dump(best_pipe, model_export_path)
print(f"✅ Selected Model '{best_model_name}' successfully exported to: {model_export_path}")

# Verification: Reload and Test Inference
loaded_pipeline = joblib.load(model_export_path)

# Test Sample Vector (Matching Streamlit Dashboard)
test_sample = pd.DataFrame([{
    'vehicle_age_years': 4.0,
    'mileage_km': 65000.0,
    'engine_size_l': 2.0,
    'fuel_type': 'Petrol',
    'transmission': 'Automatic',
    'market_segment': 'Mid_Range',
    'condition_rating': 4,
    'previous_owners': 1
}])

predicted_price = loaded_pipeline.predict(test_sample)[0]
print(f"\n🔍 Verification Sample Input:")
display(test_sample)
print(f"🎯 Model Prediction Output: RM {predicted_price:,.2f}")